In [1]:
import pandas as pd
import pyomo.environ as pyo
from food_security import optimizer

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'OneDrive - Stichting Deltares/Documents - International Delta Toolset/Food security'

In [3]:
input_file = src_dir / 'input_all.xlsx'

crop_df = pd.read_excel(input_file, sheet_name='crop_id')
trade_df = pd.read_excel(input_file, sheet_name='trade')
production_df = pd.read_excel(input_file, sheet_name='production_d')
production_df = production_df[production_df['year'] == 2020]
price_df = pd.read_excel(input_file, sheet_name='market_price')
price_df = price_df.fillna(180)
theta_df = pd.read_excel(input_file, sheet_name='plant_based_diet_goal')
energy_df = pd.read_excel(input_file, sheet_name='calories')
income_df = pd.read_excel(input_file, sheet_name='income')
price_goal_df = pd.read_excel(input_file, sheet_name='price_goal')

population = 106540000

In [4]:
income_df = income_df.merge(price_goal_df, on='income_quintiles')
income_df['p_spent_food_max'] = income_df['p_spent_food_max'] / 100

price_df = price_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["usd_per_tonnes"].mean()

production_df = production_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["yield"].sum()

trade_df = trade_df[trade_df['crop_name_fao'].isin(production_df['crop_name_fao'])]

energy_df = energy_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["calories_kcal_kg"].mean()

input_df = (
    production_df
    .merge(price_df, on="crop_name_fao", how="left")
    .merge(trade_df, on="crop_name_fao", how="left")
    .merge(energy_df, on="crop_name_fao", how="left")
)

# Convert trade from tonnes to kg
input_df["import"] = input_df["import"] * 1000
input_df["export"] = input_df["export"] * 1000

input_df["quantity"] = input_df["yield"] + input_df["import"] - input_df["export"]

class_map = (
    crop_df
    .drop_duplicates(subset=["crop_name_fao"])
    .set_index("crop_name_fao")["class"]
)

input_df["class"] = input_df["crop_name_fao"].map(class_map)
input_df = input_df[input_df['class'] != '-']
input_df["quantity"] = input_df['quantity'].clip(lower=0, upper=None)
# Convert to daily availability
input_df["quantity"] = input_df["quantity"] / 365.25
input_df["usd_per_kg"] = input_df["usd_per_tonnes"] / 1000

In [5]:
FSOptimizer = optimizer.FSOptimizer(input_df=input_df, income_df=income_df, population=population, MDER=1806.61, lambda_g={
            g: 1.0 for g in ["carb", "protein", "fat", "fruitvege"]
        })
results = FSOptimizer.solve()

In [6]:
results[0]

,5,4,3,2,1
Barley,1.896976e+05,2.095827e+05,6.302644e+04,2.534286e+04,1.556153e+04
"Beans, dry",0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Cabbages,7.435210e+06,4.832990e-01,1.494653e-02,5.133700e-03,2.136239e-03
Chick-peas,1.462113e+04,8.243385e-03,2.806646e-03,1.210927e-03,5.560190e-04
Cucumbers and gherkins,3.904230e+05,7.116038e-01,1.929863e-02,6.102043e-03,2.467498e-03
Eggplants (aubergines),1.259554e+07,4.352147e-01,1.474112e-02,5.076954e-03,2.114734e-03
"Groundnuts, Shelled",2.885960e+05,5.127780e-03,1.740411e-03,7.515887e-04,3.452668e-04
Lemons and limes,2.406574e+07,6.182976e-01,1.519690e-02,5.209661e-03,2.166325e-03
Linseed,9.597946e+04,5.910690e-03,2.009998e-03,8.676673e-04,3.985113e-04
Maize (incl. 0067 and 0068),2.192319e+05,6.273040e+06,5.184874e+06,7.604562e+05,9.088464e+04


In [7]:
results[1]

,5,4,3,2,1
Barley,0.008903,0.009836,0.002958,0.001189,0.000730
"Beans, dry",0.000000,0.000000,0.000000,0.000000,0.000000
Cabbages,0.348940,0.000000,0.000000,0.000000,0.000000
Chick-peas,0.000686,0.000000,0.000000,0.000000,0.000000
Cucumbers and gherkins,0.018323,0.000000,0.000000,0.000000,0.000000
Eggplants (aubergines),0.591118,0.000000,0.000000,0.000000,0.000000
"Groundnuts, Shelled",0.013544,0.000000,0.000000,0.000000,0.000000
Lemons and limes,1.129423,0.000000,0.000000,0.000000,0.000000
Linseed,0.004504,0.000000,0.000000,0.000000,0.000000
Maize (incl. 0067 and 0068),0.010289,0.294398,0.243330,0.035689,0.004265


In [8]:
results[2]

,5,4,3,2,1
objective function,0.372729,1.840000,1.840000,1.840000,1.840000
spending_pp,1.580570,0.282804,0.240232,0.229932,0.226192
percentage_spending,0.131714,0.031423,0.030029,0.038322,0.056548
consumed_energy_pp,1806.610018,1807.124649,1806.610133,1806.610000,1807.995136
carb_fraction,0.613131,1.000000,1.000000,1.000000,1.000000
fat_fraction,0.071085,0.000000,0.000000,0.000000,0.000000
fruitvege_fraction,0.306467,0.000000,0.000000,0.000000,0.000000
protein_fraction,0.009318,0.000000,0.000000,0.000000,0.000000
carb_kg,0.574074,0.603103,0.633270,0.727633,0.739047
fat_kg,0.024016,0.000000,0.000000,0.000000,0.000000
